In [ ]:
from pathlib import Path
import json, subprocess, sys, time
import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
folders = [
    p for p in INPUT.rglob("nuts_validation_inputs")
    if p.is_dir() and all((p / name).is_file() for name in
                          ("binary.csv", "zeeman.csv", "known_truth.json"))
]
if not folders:
    raise FileNotFoundError(
        "Attach the saved Day 7 Physics notebook output containing "
        "nuts_validation_inputs/binary.csv, zeeman.csv, and known_truth.json."
    )
folders.sort(key=lambda p: p.stat().st_mtime, reverse=True)
VALIDATION_DIR = folders[0]
print("Validation folder:", VALIDATION_DIR, "| complete folders:", len(folders))

try:
    import jax
    import numpyro
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpyro>=0.15"], check=True)
    import jax
    import numpyro
if jax.default_backend() != "gpu":
    raise RuntimeError(f"JAX backend is {jax.default_backend()}; select GPU T4 x2 and restart the Kaggle session.")
DEVICES = jax.devices()
print("JAX:", jax.default_backend(), DEVICES, "| NumPyro:", numpyro.__version__)

binary_df = pd.read_csv(VALIDATION_DIR / "binary.csv")
zeeman_df = pd.read_csv(VALIDATION_DIR / "zeeman.csv")
TRUTH = json.loads((VALIDATION_DIR / "known_truth.json").read_text(encoding="utf-8"))
OUT = WORK / "day8-1-map-initialized"
OUT.mkdir(parents=True, exist_ok=True)
assert len(binary_df) >= 100 and len(zeeman_df) >= 100
print("[PASS] rows: binary =", len(binary_df), "| Zeeman =", len(zeeman_df))
print("[PASS] output:", OUT)
print("Preflight only: no fitting has started.")


In [ ]:
from scipy.optimize import minimize
from astropy.timeseries import BoxLeastSquares

# Smooth approximation to clip(x, 0, 1), used to reduce sharp NUTS gradients.
def soft_clip_np(x, sharpness=80.0):
    return (np.logaddexp(0.0, sharpness*x) - np.logaddexp(0.0, sharpness*(x-1.0))) / sharpness

def soft_clip_jax(x, sharpness=80.0):
    import jax.nn as jnn
    return (jnn.softplus(sharpness*x) - jnn.softplus(sharpness*(x-1.0))) / sharpness

# Binary input normalization and BLS local mode.
bt = binary_df["time_days"].to_numpy(float)
bf = binary_df["flux"].to_numpy(float)
be = binary_df["flux_err"].to_numpy(float)
ok = np.isfinite(bt) & np.isfinite(bf) & np.isfinite(be) & (be > 0)
bt, bf, be = bt[ok], bf[ok], be[ok]
order = np.argsort(bt)
bt, bf, be = bt[order], bf[order], be[order]
bmed = np.median(bf)
bf, be = bf/bmed, be/bmed
bt -= bt.min()
max_period = min(13.0, np.ptp(bt)/2.0)
bls = BoxLeastSquares(bt, bf, dy=be).autopower(
    duration=np.geomspace(0.005, min(0.09, max_period*0.12), 14),
    objective="snr", minimum_period=0.1, maximum_period=max_period, frequency_factor=20.0,
)
bi = int(np.nanargmax(bls.power))
BLS_PERIOD = float(bls.period[bi])
BLS_PHASE = float(np.mod(bls.transit_time[bi]/BLS_PERIOD, 1.0))
BLS_DEPTH = float(np.clip(bls.depth[bi], .03, .35))
BLS_DUR = float(np.clip(bls.duration[bi]/BLS_PERIOD, .02, .12))
print(f"BLS start: period={BLS_PERIOD:.6f} d, phase={BLS_PHASE:.5f}, duration={BLS_DUR:.5f}")

def binary_predict_np(t, p, phase, depth, ratio, duration, ellip, refl):
    orb = np.mod(t/p - phase + .5, 1.0) - .5
    half = duration/2.0
    ingress = half*.18
    primary = soft_clip_np((half+ingress-np.abs(orb))/(2*ingress))
    secondary_distance = .5-np.abs(orb)
    secondary = soft_clip_np((half+ingress-np.abs(secondary_distance))/(2*ingress))
    return 1.0-depth*primary-depth*ratio*secondary + ellip*np.cos(4*np.pi*orb) + refl*np.cos(2*np.pi*orb)

# MAP vector: period, phase, primary depth, depth ratio, duration fraction,
# ellipsoidal amplitude, reflection amplitude, log(extra noise).
def binary_objective(x):
    p, phase, depth, ratio, duration, ellip, refl, log_noise = x
    noise = np.exp(log_noise)
    pred = binary_predict_np(bt, p, phase, depth, ratio, duration, ellip, refl)
    sigma = np.sqrt(be**2 + noise**2)
    nll = .5*np.sum(((bf-pred)/sigma)**2 + np.log(2*np.pi*sigma**2))
    # Same local priors used by the Bayesian model; constants independent of x omitted.
    nll += .5*((p-BLS_PERIOD)/.012)**2 + .5*((phase-BLS_PHASE)/.025)**2
    nll += .5*(ellip/.01)**2 + .5*(refl/.01)**2 + .5*(noise/.01)**2 - log_noise
    return float(nll) if np.isfinite(nll) else 1e100

binary_bounds=[
    (max(.1,BLS_PERIOD-.04), BLS_PERIOD+.04),
    (BLS_PHASE-.05, BLS_PHASE+.05),
    (.03,.35), (0.,1.), (.02,.12), (-.02,.02), (-.02,.02),
    (np.log(1e-4),np.log(.03))
]
binary_starts=[]
for dp in (-.015,0.0,.015):
    for dh in (-.02,0.0,.02):
        for dur in (BLS_DUR,.08):
            binary_starts.append([
                np.clip(BLS_PERIOD+dp,*binary_bounds[0]),
                np.clip(BLS_PHASE+dh,*binary_bounds[1]),
                BLS_DEPTH,.45,dur,0.0,0.0,np.log(.002)
            ])
binary_fits=[minimize(binary_objective,x0,method="L-BFGS-B",bounds=binary_bounds,
                      options={"maxiter":1200,"ftol":1e-11}) for x0 in binary_starts]
binary_map=min(binary_fits,key=lambda r:r.fun)
if not binary_map.success and not np.isfinite(binary_map.fun):
    raise RuntimeError("Binary multi-start MAP optimization failed.")
binary_map_values=binary_map.x
print("Best binary MAP objective:", round(float(binary_map.fun),3),
      "| optimizer:", binary_map.message)
print("Binary MAP:", dict(zip(
    ["period_days","phase","depth_primary","depth_ratio","duration_fraction",
     "ellipsoidal_amplitude","reflection_amplitude","extra_noise"],
    [float(binary_map_values[0]),float(binary_map_values[1]),float(binary_map_values[2]),
     float(binary_map_values[3]),float(binary_map_values[4]),float(binary_map_values[5]),
     float(binary_map_values[6]),float(np.exp(binary_map_values[7]))]
)))

# Zeeman input: fit a local H-alpha window, with field in kG and velocity in 50 km/s units.
REST_WAVE=6562.8
zw=zeeman_df["wavelength_angstrom"].to_numpy(float)
zf=zeeman_df["normalized_flux"].to_numpy(float)
ze=zeeman_df["flux_err"].to_numpy(float)
ok=np.isfinite(zw)&np.isfinite(zf)&np.isfinite(ze)&(ze>0)
zw,zf,ze=zw[ok],zf[ok],ze[ok]
local=np.abs(zw-REST_WAVE)<=8.0
zw,zf,ze=zw[local],zf[local],ze[local]
if len(zw)<50: raise RuntimeError(f"Only {len(zw)} samples in H-alpha window.")
def zeeman_predict_np(wave, field_kg, velocity_50, depth, width, continuum):
    velocity=velocity_50*50.0
    shifted=REST_WAVE*(1.0+velocity/299792.458)
    delta=4.67e-13*shifted**2*(field_kg*1000.0)
    pi=np.exp(-.5*((wave-shifted)/width)**2)
    blue=np.exp(-.5*((wave-(shifted-delta))/width)**2)
    red=np.exp(-.5*((wave-(shifted+delta))/width)**2)
    return 1.0+continuum-depth*(pi+blue+red)/3.0

def zeeman_objective(x):
    field_kg,velocity_50,depth,width,continuum,log_noise=x
    noise=np.exp(log_noise)
    pred=zeeman_predict_np(zw,field_kg,velocity_50,depth,width,continuum)
    sigma=np.sqrt(ze**2+noise**2)
    velocity=velocity_50*50.0
    nll=.5*np.sum(((zf-pred)/sigma)**2+np.log(2*np.pi*sigma**2))
    nll+=.5*(velocity/100.0)**2+.5*(continuum/.01)**2+.5*(noise/.01)**2-log_noise
    return float(nll) if np.isfinite(nll) else 1e100

zeeman_bounds=[(0.,50.),(-5.,5.),(.1,.85),(.25,1.5),(-.05,.05),(np.log(1e-4),np.log(.04))]
zeeman_starts=[]
for field0 in (5.,25.,40.):
    for velocity0 in (-1.5,0.,1.5):
        zeeman_starts.append([field0,velocity0,.6,.5,0.,np.log(.002)])
zeeman_fits=[minimize(zeeman_objective,x0,method="L-BFGS-B",bounds=zeeman_bounds,
                      options={"maxiter":1500,"ftol":1e-11}) for x0 in zeeman_starts]
zeeman_map=min(zeeman_fits,key=lambda r:r.fun)
if not np.isfinite(zeeman_map.fun): raise RuntimeError("Zeeman multi-start MAP optimization failed.")
zeeman_map_values=zeeman_map.x
print("Best Zeeman MAP objective:",round(float(zeeman_map.fun),3),
      "| optimizer:",zeeman_map.message)
print("Zeeman MAP:",{
    "field_gauss":float(zeeman_map_values[0]*1000),
    "radial_velocity_km_s":float(zeeman_map_values[1]*50),
    "line_depth":float(zeeman_map_values[2]),
    "line_sigma_angstrom":float(zeeman_map_values[3]),
    "continuum_offset":float(zeeman_map_values[4]),
    "intrinsic_noise":float(np.exp(zeeman_map_values[5]))
})

In [ ]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC,NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary

WARMUP=500
SAMPLES=800
CHAINS=2
CHAIN_METHOD="parallel" if len(DEVICES)>=CHAINS else "sequential"
NUTS_RESULTS={}

bt_j,bf_j,be_j=map(jnp.asarray,(bt,bf,be))
def binary_model():
    period=numpyro.sample("period_days",dist.TruncatedNormal(
        BLS_PERIOD,.012,low=max(.1,BLS_PERIOD-.04),high=BLS_PERIOD+.04))
    phase_offset=numpyro.sample("phase_offset",dist.Normal(0.,.025))
    phase=BLS_PHASE+phase_offset
    depth=numpyro.sample("depth_primary",dist.Uniform(.03,.35))
    ratio=numpyro.sample("depth_ratio",dist.Uniform(0.,1.))
    duration=numpyro.sample("duration_fraction",dist.Uniform(.02,.12))
    ellip=numpyro.sample("ellipsoidal_amplitude",dist.Normal(0.,.01))
    refl=numpyro.sample("reflection_amplitude",dist.Normal(0.,.01))
    noise_unit=numpyro.sample("extra_noise_unit",dist.HalfNormal(5.))
    noise=noise_unit*.002
    orb=jnp.mod(bt_j/period-phase+.5,1.)-.5
    half=duration/2.
    ingress=half*.18
    primary=soft_clip_jax((half+ingress-jnp.abs(orb))/(2*ingress))
    secondary_distance=.5-jnp.abs(orb)
    secondary=soft_clip_jax((half+ingress-jnp.abs(secondary_distance))/(2*ingress))
    pred=1.-depth*primary-depth*ratio*secondary
    pred+=ellip*jnp.cos(4*jnp.pi*orb)+refl*jnp.cos(2*jnp.pi*orb)
    sigma=jnp.sqrt(be_j**2+noise**2)
    numpyro.sample("flux",dist.Normal(pred,sigma),obs=bf_j)

binary_init={
    "period_days":float(binary_map_values[0]),
    "phase_offset":float(binary_map_values[1]-BLS_PHASE),
    "depth_primary":float(binary_map_values[2]),
    "depth_ratio":float(binary_map_values[3]),
    "duration_fraction":float(binary_map_values[4]),
    "ellipsoidal_amplitude":float(binary_map_values[5]),
    "reflection_amplitude":float(binary_map_values[6]),
    "extra_noise_unit":float(np.exp(binary_map_values[7])/.002),
}

zw_j,zf_j,ze_j=map(jnp.asarray,(zw,zf,ze))
def zeeman_model():
    field_kg=numpyro.sample("field_kilogauss",dist.Uniform(0.,50.))
    velocity_50=numpyro.sample("velocity_50kms",dist.Normal(0.,2.))
    depth=numpyro.sample("line_depth",dist.Uniform(.1,.85))
    width=numpyro.sample("line_sigma_angstrom",dist.Uniform(.25,1.5))
    continuum=numpyro.sample("continuum_offset",dist.Normal(0.,.01))
    noise_unit=numpyro.sample("intrinsic_noise_unit",dist.HalfNormal(5.))
    noise=noise_unit*.002
    velocity=velocity_50*50.
    shifted=REST_WAVE*(1.+velocity/299792.458)
    delta=4.67e-13*shifted**2*(field_kg*1000.)
    pi=jnp.exp(-.5*((zw_j-shifted)/width)**2)
    blue=jnp.exp(-.5*((zw_j-(shifted-delta))/width)**2)
    red=jnp.exp(-.5*((zw_j-(shifted+delta))/width)**2)
    pred=1.+continuum-depth*(pi+blue+red)/3.
    sigma=jnp.sqrt(ze_j**2+noise**2)
    numpyro.sample("spectrum",dist.Normal(pred,sigma),obs=zf_j)

def sample_family(name,model,init,seed):
    started=time.time()
    try:
        kernel=NUTS(model,target_accept_prob=.95,max_tree_depth=9,
                    init_strategy=init_to_value(values=init))
        mcmc=MCMC(kernel,num_warmup=WARMUP,num_samples=SAMPLES,num_chains=CHAINS,
                  chain_method=CHAIN_METHOD,progress_bar=True)
        mcmc.run(jax.random.PRNGKey(seed))
        grouped=mcmc.get_samples(group_by_chain=True)
        flat=mcmc.get_samples(group_by_chain=False)
        stats=numpyro_summary(grouped,group_by_chain=True)
        per_parameter={}
        for key,values in stats.items():
            per_parameter[key]={}
            for metric in ("mean","std","median","5%","95%","n_eff","r_hat"):
                if metric in values:
                    arr=np.asarray(values[metric])
                    if arr.ndim==0 and np.isfinite(arr):
                        per_parameter[key][metric]=float(arr)
        rhats=[v["r_hat"] for v in per_parameter.values() if "r_hat" in v]
        esses=[v["n_eff"] for v in per_parameter.values() if "n_eff" in v]
        divergent=int(np.asarray(mcmc.get_extra_fields(group_by_chain=True)["diverging"]).sum())
        np.savez_compressed(OUT/f"{name}_posterior_draws.npz",**{k:np.asarray(v) for k,v in flat.items()})
        record={
            "family":name,"status":"screen_pass" if rhats and max(rhats)<=1.05 and esses and min(esses)>=100 and divergent==0 else "needs_review",
            "seconds":round(time.time()-started,1),"max_r_hat":max(rhats) if rhats else None,
            "min_ess":min(esses) if esses else None,"divergences":divergent,
            "per_parameter_diagnostics":per_parameter,
            "samples":{k:np.asarray(v) for k,v in flat.items()},
        }
    except Exception as exc:
        record={"family":name,"status":"failed","error":f"{type(exc).__name__}: {exc}",
                "seconds":round(time.time()-started,1),"per_parameter_diagnostics":{}}
    NUTS_RESULTS[name]=record
    print(f"[{record['status']}] {name}: {record.get('seconds')}s, "
          f"max R-hat={record.get('max_r_hat')}, min ESS={record.get('min_ess')}, "
          f"divergences={record.get('divergences')}",flush=True)

sample_family("binary",binary_model,binary_init,9041)
sample_family("zeeman",zeeman_model,{
    "field_kilogauss":float(zeeman_map_values[0]),
    "velocity_50kms":float(zeeman_map_values[1]),
    "line_depth":float(zeeman_map_values[2]),
    "line_sigma_angstrom":float(zeeman_map_values[3]),
    "continuum_offset":float(zeeman_map_values[4]),
    "intrinsic_noise_unit":float(np.exp(zeeman_map_values[5])/.002),
},9042)

In [ ]:
import matplotlib.pyplot as plt

def qrow(family,param,draws,truth,unit=""):
    x=np.asarray(draws,float)
    lo,med,hi=np.quantile(x,[.05,.5,.95])
    return {"family":family,"parameter":param,"unit":unit,"q05":float(lo),
            "median":float(med),"q95":float(hi),
            "known_truth":None if truth is None else float(truth),
            "truth_inside_90pct_interval":None if truth is None else bool(lo<=truth<=hi)}

records=[]
truth_b=TRUTH.get("binary",{})
truth_z=TRUTH.get("zeeman",{})
bresult=NUTS_RESULTS.get("binary",{})
zresult=NUTS_RESULTS.get("zeeman",{})
if "samples" in bresult:
    s=bresult["samples"]
    records += [
        qrow("binary","period_days",s["period_days"],truth_b.get("period_days"),"day"),
        qrow("binary","phase",np.mod(BLS_PHASE+s["phase_offset"],1.),truth_b.get("phase"),"cycle"),
        qrow("binary","depth_primary",s["depth_primary"],truth_b.get("depth_primary")),
        qrow("binary","depth_ratio",s["depth_ratio"],truth_b.get("depth_ratio")),
        qrow("binary","duration_fraction",s["duration_fraction"],truth_b.get("duration_fraction"),"cycle"),
        qrow("binary","extra_noise",s["extra_noise_unit"]*.002,truth_b.get("extra_noise")),
    ]
if "samples" in zresult:
    s=zresult["samples"]
    records += [
        qrow("zeeman","field_gauss",s["field_kilogauss"]*1000.,truth_z.get("field_gauss"),"gauss"),
        qrow("zeeman","radial_velocity_km_s",s["velocity_50kms"]*50.,truth_z.get("radial_velocity_km_s"),"km/s"),
        qrow("zeeman","line_depth",s["line_depth"],truth_z.get("line_depth")),
        qrow("zeeman","line_sigma_angstrom",s["line_sigma_angstrom"],truth_z.get("line_sigma_angstrom"),"angstrom"),
        qrow("zeeman","intrinsic_noise",s["intrinsic_noise_unit"]*.002,truth_z.get("intrinsic_noise")),
    ]
intervals=pd.DataFrame(records)
intervals.to_csv(OUT/"day8_1_posterior_recovery_intervals.csv",index=False)
pd.DataFrame([
    {"family":family,"parameter":param,**metrics}
    for family,r in NUTS_RESULTS.items()
    for param,metrics in r.get("per_parameter_diagnostics",{}).items()
]).to_csv(OUT/"day8_1_per_parameter_diagnostics.csv",index=False)

fig,ax=plt.subplots(1,2,figsize=(14,5))
if "samples" in bresult:
    s=bresult["samples"]
    med={k:float(np.median(v)) for k,v in s.items()}
    tgrid=np.linspace(bt.min(),bt.max(),5000)
    ph=float(np.mod(BLS_PHASE+med["phase_offset"],1.))
    curve=binary_predict_np(tgrid,med["period_days"],ph,med["depth_primary"],med["depth_ratio"],
                            med["duration_fraction"],med["ellipsoidal_amplitude"],med["reflection_amplitude"])
    ax[0].scatter(bt,bf,s=5,alpha=.3,label="synthetic data")
    ax[0].plot(tgrid,curve,color="crimson",lw=1.2,label="posterior median")
    ax[0].set(title="Binary local-mode fit",xlabel="Time (days)",ylabel="Normalized flux")
    ax[0].legend()
else: ax[0].set_visible(False)
if "samples" in zresult:
    s=zresult["samples"]
    med={k:float(np.median(v)) for k,v in s.items()}
    wgrid=np.linspace(zw.min(),zw.max(),1200)
    curve=zeeman_predict_np(wgrid,med["field_kilogauss"],med["velocity_50kms"],
                            med["line_depth"],med["line_sigma_angstrom"],med["continuum_offset"])
    ax[1].scatter(zw,zf,s=8,alpha=.4,label="synthetic data")
    ax[1].plot(wgrid,curve,color="crimson",lw=1.3,label="posterior median")
    ax[1].set(title="Zeeman H-alpha fit",xlabel="Wavelength (Å)",ylabel="Normalized flux")
    ax[1].legend()
else: ax[1].set_visible(False)
fig.tight_layout()
fig.savefig(OUT/"day8_1_posterior_fits.png",dpi=160)
plt.show()

binary_map_prediction=binary_predict_np(
    bt,binary_map_values[0],binary_map_values[1],binary_map_values[2],binary_map_values[3],
    binary_map_values[4],binary_map_values[5],binary_map_values[6]
)
zeeman_map_prediction=zeeman_predict_np(
    zw,zeeman_map_values[0],zeeman_map_values[1],zeeman_map_values[2],
    zeeman_map_values[3],zeeman_map_values[4]
)
report={
    "created_utc":pd.Timestamp.now(tz="UTC").isoformat(),
    "validation_input_dir":str(VALIDATION_DIR),
    "jax_backend":jax.default_backend(),
    "map_initialization":{
        "binary":{"objective":float(binary_map.fun),"success":bool(binary_map.success),
                  "period_days":float(binary_map_values[0]),"phase":float(binary_map_values[1]),
                  "duration_fraction":float(binary_map_values[4]),
                  "normalized_flux_rmse":float(np.sqrt(np.mean((bf-binary_map_prediction)**2)))},
        "zeeman":{"objective":float(zeeman_map.fun),"success":bool(zeeman_map.success),
                  "normalized_flux_rmse":float(np.sqrt(np.mean((zf-zeeman_map_prediction)**2))),
                  "field_gauss":float(zeeman_map_values[0]*1000),
                  "radial_velocity_km_s":float(zeeman_map_values[1]*50),
                  "line_sigma_angstrom":float(zeeman_map_values[3])}
    },
    "nuts_results":{k:{key:value for key,value in r.items() if key!="samples"} for k,r in NUTS_RESULTS.items()},
    "files":{
        "intervals":str(OUT/"day8_1_posterior_recovery_intervals.csv"),
        "parameter_diagnostics":str(OUT/"day8_1_per_parameter_diagnostics.csv"),
        "fit_plot":str(OUT/"day8_1_posterior_fits.png"),
        "posterior_draws":[str(OUT/"binary_posterior_draws.npz"),str(OUT/"zeeman_posterior_draws.npz")]
    },
    "limits":[
        "Known-truth synthetic examples only; not real-star inference.",
        "Binary inference is conditional on a BLS-localized period mode.",
        "Zeeman inference assumes identified H-alpha rest wavelength 6562.8 Angstrom.",
        "Two-chain diagnostics are screening only; independent runs and posterior predictive checks are required before claiming validated inference."
    ]
}
(OUT/"day8_1_bayesian_recovery_summary.json").write_text(json.dumps(report,indent=2,default=str),encoding="utf-8")
print(intervals.to_string(index=False))
print("\nNUTS run status:")
print(pd.DataFrame([{"family":k,"status":v.get("status"),"max_r_hat":v.get("max_r_hat"),
                     "min_ess":v.get("min_ess"),"divergences":v.get("divergences")}
                    for k,v in NUTS_RESULTS.items()]).to_string(index=False))
print("\nSaved report:",OUT/"day8_1_bayesian_recovery_summary.json")
print("All outputs:",OUT)
